In [ ]:
!pip install -q transformers accelerate peft bitsandbytes
!pip install -q sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [ ]:
base_model = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded


In [ ]:
model = PeftModel.from_pretrained(
    model,
    "/content/adapter_model"
)

print("Adapter loaded")

Adapter loaded


In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

print("Merged model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved


In [ ]:
!mkdir quantized

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config,
    device_map="auto"
)

model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")

print("INT8 quantized model saved")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 quantized model saved


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_config,
    device_map="auto"
)

model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")

print("INT4 quantized model saved")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 quantized model saved


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make

Cloning into 'llama.cpp'...
remote: Enumerating objects: 82446, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 82446 (delta 45), reused 16 (delta 16), pack-reused 82345 (from 3)
Receiving objects: 100% (82446/82446), 312.74 MiB | 24.15 MiB/s, done.
Resolving deltas: 100% (59279/59279), done.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
/content/merged_model \
--outfile /content/quantized/model-f16.gguf \
--outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16 --

In [ ]:
!du -sh /content/merged_model
!du -sh /content/quantized/model-int8
!du -sh /content/quantized/model-int4
!du -sh /content/quantized/model-f16.gguf

2.9G	/content/merged_model
1.7G	/content/quantized/model-int8
1.2G	/content/quantized/model-int4
2.9G	/content/quantized/model-f16.gguf


In [ ]:
!ls -lh /content/quantized

total 2.9G
-rw-r--r-- 1 root root 2.9G Mar  7 20:11 model-f16.gguf
drwxr-xr-x 2 root root 4.0K Mar  7 20:02 model-int4
drwxr-xr-x 2 root root 4.0K Mar  7 20:00 model-int8


In [ ]:
import os
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

models = {
    "FP16": "/content/merged_model",
    "INT8": "/content/quantized/model-int8",
    "INT4": "/content/quantized/model-int4"
}

prompt = "Explain machine learning in simple terms."

results = []

for name, path in models.items():

    print(f"\nTesting {name} model...\n")

    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path).to("cuda")

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

    end = time.time()

    inference_time = end - start

    tokens_generated = outputs.shape[1]

    speed = tokens_generated / inference_time

    size = os.popen(f'du -sh {path}').read().split()[0]

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(text)

    results.append({
        "Model": name,
        "Size": size,
        "Time (s)": round(inference_time,2),
        "Tokens/sec": round(speed,2)
    })

print("\nFinal Benchmark Results\n")

for r in results:
    print(r)


Testing FP16 model...



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Explain machine learning in simple terms. Machine learning is a type of artificial intelligence that allows computers to learn and improve their performance on a specific task without being explicitly programmed. It involves using algorithms and statistical models to analyze large amounts of data and identify patterns or relationships between variables, which can then

Testing INT8 model...



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Explain machine learning in simple terms. Machine Learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms and statistical models to analyze data, identify patterns or trends, and make predictions or decisions based on that analysis. In other

Testing INT4 model...



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Explain machine learning in simple terms. Machine Learning is a branch of artificial intelligence that allows computers to learn from data and improve their performance on specific tasks over time without explicit programming.

In other words, ML algorithms can automatically "learn" patterns and relationships within large datasets by analyzing the data and

Final Benchmark Results

{'Model': 'FP16', 'Size': '2.9G', 'Time (s)': 2.49, 'Tokens/sec': 23.25}
{'Model': 'INT8', 'Size': '1.7G', 'Time (s)': 8.99, 'Tokens/sec': 6.45}
{'Model': 'INT4', 'Size': '1.2G', 'Time (s)': 3.52, 'Tokens/sec': 16.47}


In [ ]:
!cd /content/llama.cpp && cmake -B build
!cd /content/llama.cpp && cmake --build build --config Release -j4

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
!./llama.cpp/build/bin/llama-cli -m /content/quantized/model-f16.gguf -p "Explain machine learning in simple terms." -n 50

AGENTS.md		       convert_lora_to_gguf.py	pocs
AUTHORS			       docs			poetry.lock
benches			       examples			pyproject.toml
build-xcframework.sh	       flake.lock		pyrightconfig.json
ci			       flake.nix		README.md
CLAUDE.md		       ggml			requirements
cmake			       gguf-py			requirements.txt
CMakeLists.txt		       grammars			scripts
CMakePresets.json	       include			SECURITY.md
CODEOWNERS		       LICENSE			src
common			       licenses			tests
CONTRIBUTING.md		       Makefile			tools
convert_hf_to_gguf.py	       media			vendor
convert_hf_to_gguf_update.py   models
convert_llama_ggml_to_gguf.py  mypy.ini


In [ ]:
import shutil
from google.colab import files

# Zip the quantized folder
shutil.make_archive("quantized_models", "zip", "/content/quantized")

# Download the zip file
files.download("quantized_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil

shutil.copytree("/content/quantized", "/content/drive/MyDrive/quantized_models")

'/content/drive/MyDrive/quantized_models'

In [1]:
!./llama.cpp/build/bin/llama-quantize \
/content/quantized/model-f16.gguf \
/content/quantized/model-q8_0.gguf \
q8_0

/bin/bash: line 1: ./llama.cpp/build/bin/llama-quantize: No such file or directory


In [3]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 82612, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 82612 (delta 100), reused 30 (delta 29), pack-reused 82421 (from 4)
Receiving objects: 100% (82612/82612), 313.26 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (59416/59416), done.


In [4]:
%cd /content/llama.cpp

!cmake -B build
!cmake --build build --config Release -j4

/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend


In [6]:
!./build/bin/llama-quantize \
/content/drive/MyDrive/quantized_models/model-f16.gguf \
/content/drive/MyDrive/quantized_models/model-q8_0.gguf \
q8_0

main: build = 8248 (5f4cdac38)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/drive/MyDrive/quantized_models/model-f16.gguf' to '/content/drive/MyDrive/quantized_models/model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/drive/MyDrive/quantized_models/model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv  

In [7]:
!./build/bin/llama-cli \
-m /content/drive/MyDrive/quantized_models/model-q8_0.gguf \
-p "Explain machine learning in simple terms." \
-n 50


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8248-5f4cdac38
model      : model-q8_0.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain machine learning in simple terms.

|-\|/-\|/-\|/-\|/-\|/-\|/-\ Machine learning is a type of artificial intelligence that allows computers to learn and improve their performance without being explicitly programmed. It involves using algorithms and statistical models t